In [1]:
import ee
import pandas as pd
import numpy as np
import time
from tqdm import tqdm

import warnings
warnings.filterwarnings('ignore')

In [2]:
try:
    service_account = 'your service account'
    credentials = ee.ServiceAccountCredentials(service_account, 'your path to json')
    ee.Initialize(credentials, project='your GEE project name')
    print("Подключение к Earth Engine успешно")

except Exception as e:
    print(f"Ошибка подключения: {e}")

Подключение к Earth Engine успешно


In [3]:
lucas_dataset = 'LUCAS-SOIL-2018.csv'

In [6]:
class FeatureExtractor:

    def __init__(self):
        self.scale = 100

    def get_sentinel_bands(self, point):
        """Извлечение спектральных каналов Sentinel-2"""
        bands_dict = {}

        try:
            collection = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
                         .filterBounds(point)
                         .filterDate('2018-06-01', '2018-08-31')
                         .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 50)))

            composite = collection.median().select(['B2', 'B3', 'B4', 'B8', 'B11', 'B12'])

            # Используем reduceRegion для извлечения значения в точке
            for band in ['B2', 'B3', 'B4', 'B8', 'B11', 'B12']:
                value = composite.select(band).reduceRegion(
                    reducer=ee.Reducer.first(),
                    geometry=point,
                    scale=self.scale,
                    bestEffort=True
                ).get(band)

                if value:
                    bands_dict[band] = value.getInfo()
                else:
                    bands_dict[band] = np.nan

        except Exception as e:
            print(f"Ошибка при извлечении Sentinel-2 данных: {e}")
            for band in ['B2', 'B3', 'B4', 'B8', 'B11', 'B12']:
                bands_dict[band] = np.nan

        return bands_dict

    def get_indices(self, point):
        """Извлечение спектральных индексов"""
        indices_dict = {}

        try:
            collection = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
                         .filterBounds(point)
                         .filterDate('2018-06-01', '2018-08-31')
                         .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 50)))

            composite = collection.median()

            # BSF (Burned Area Index)
            nbr2 = composite.normalizedDifference(['B11', 'B12'])
            bsf = nbr2.lt(0).rename('BSF')

            bsf_value = bsf.reduceRegion(
                reducer=ee.Reducer.first(),
                geometry=point,
                scale=self.scale,
                bestEffort=True
            ).get('BSF')

            indices_dict['BSF'] = bsf_value.getInfo() if bsf_value else np.nan

            # CAL_mask - маска облачности (используем существующую маску)
            qa = collection.select('QA60').median()
            cal_mask = qa.reduceRegion(
                reducer=ee.Reducer.first(),
                geometry=point,
                scale=self.scale,
                bestEffort=True
            ).get('QA60')

            indices_dict['CAL_mask'] = cal_mask.getInfo() if cal_mask else np.nan

            # NDVI как дополнительный индекс
            ndvi = composite.normalizedDifference(['B8', 'B4']).rename('NDVI')
            ndvi_value = ndvi.reduceRegion(
                reducer=ee.Reducer.first(),
                geometry=point,
                scale=self.scale,
                bestEffort=True
            ).get('NDVI')

            indices_dict['NDVI'] = ndvi_value.getInfo() if ndvi_value else np.nan

            # ZPC1 - временно используем NDVI как прокси
            indices_dict['ZPC1'] = indices_dict['NDVI']

        except Exception as e:
            print(f"Ошибка при извлечении индексов: {e}")
            indices_dict['BSF'] = np.nan
            indices_dict['CAL_mask'] = np.nan
            indices_dict['ZPC1'] = np.nan
            indices_dict['NDVI'] = np.nan

        return indices_dict

    def get_climate_data(self, point):
        """Извлечение климатических данных"""
        climate_dict = {}

        try:
            # WorldClim биоклиматические переменные
            worldclim = ee.Image("WORLDCLIM/V1/BIO")

            # Функция для безопасного извлечения значения
            def safe_extract(image, band, point):
                try:
                    value = image.select(band).reduceRegion(
                        reducer=ee.Reducer.first(),
                        geometry=point,
                        scale=1000,
                        bestEffort=True
                    ).get(band)
                    return value.getInfo() if value else np.nan
                except:
                    return np.nan

            climate_dict['t_mean_annual'] = safe_extract(worldclim, 'bio01', point)
            climate_dict['t_max'] = safe_extract(worldclim, 'bio05', point)
            climate_dict['t_min'] = safe_extract(worldclim, 'bio06', point)
            climate_dict['precip_annual'] = safe_extract(worldclim, 'bio12', point)

            # TerraClimate для ETP и ГТК
            terra_2018 = (ee.ImageCollection('IDAHO_EPSCOR/TERRACLIMATE')
                         .filterDate('2018-01-01', '2018-12-31'))

            # ETP (эвапотранспирация) - сумма за год
            pet_2018 = terra_2018.select('pet').sum()
            etp = pet_2018.reduceRegion(
                reducer=ee.Reducer.first(),
                geometry=point,
                scale=1000,
                bestEffort=True
            ).get('pet')

            climate_dict['ETP'] = etp.getInfo() if etp else np.nan

            # ГТК (Гидротермический коэффициент Селянинова)
            temp_sum = 0
            precip_sum = 0

            for m in range(4, 10):  # апрель-сентябрь
                month_str = f'2018-{m:02d}-01'
                next_month = f'2018-{m+1:02d}-01' if m < 9 else '2018-10-01'

                month_data = terra_2018.filterDate(month_str, next_month).mean()

                # Получаем значения для месяца
                tmmx_val = month_data.select('tmmx').reduceRegion(
                    reducer=ee.Reducer.first(),
                    geometry=point,
                    scale=1000,
                    bestEffort=True
                ).get('tmmx')

                tmmn_val = month_data.select('tmmn').reduceRegion(
                    reducer=ee.Reducer.first(),
                    geometry=point,
                    scale=1000,
                    bestEffort=True
                ).get('tmmn')

                pr_val = month_data.select('pr').reduceRegion(
                    reducer=ee.Reducer.first(),
                    geometry=point,
                    scale=1000,
                    bestEffort=True
                ).get('pr')

                if tmmx_val and tmmn_val and pr_val:
                    t_mean = (tmmx_val.getInfo() + tmmn_val.getInfo()) / 2
                    pr_m = pr_val.getInfo()

                    if t_mean > 283.15:
                        temp_sum += t_mean
                        precip_sum += pr_m

            if temp_sum > 0:
                climate_dict['GTK'] = precip_sum / (temp_sum / 10)
            else:
                climate_dict['GTK'] = np.nan

        except Exception as e:
            print(f"Ошибка при извлечении климатических данных: {e}")
            for key in ['t_mean_annual', 't_max', 't_min', 'precip_annual',
                       'GTK', 'ETP']:
                climate_dict[key] = np.nan

        return climate_dict

    def get_topo_data(self, point):
        """Извлечение топографических данных"""
        topo_dict = {}

        try:
            srtm = ee.Image("USGS/SRTMGL1_003")
            topo = ee.Terrain.products(srtm)

            def safe_extract_topo(band):
                try:
                    value = topo.select(band).reduceRegion(
                        reducer=ee.Reducer.first(),
                        geometry=point,
                        scale=30,
                        bestEffort=True
                    ).get(band)
                    return value.getInfo() if value else np.nan
                except:
                    return np.nan

            # Elevation
            elev = srtm.select('elevation').reduceRegion(
                reducer=ee.Reducer.first(),
                geometry=point,
                scale=30,
                bestEffort=True
            ).get('elevation')

            topo_dict['elevation'] = elev.getInfo() if elev else np.nan
            topo_dict['slope'] = safe_extract_topo('slope')
            topo_dict['aspect'] = safe_extract_topo('aspect')

            # TWI (Topographic Wetness Index) - приближенное вычисление
            # TWI = ln(a / tan(b)), где a - площадь водосбора, b - уклон
            # Используем упрощенную версию
            slope_rad = topo.select('slope').multiply(3.14159).divide(180)
            twi = srtm.select('elevation').add(1).divide(slope_rad.add(0.001)).log()

            twi_value = twi.reduceRegion(
                reducer=ee.Reducer.first(),
                geometry=point,
                scale=30,
                bestEffort=True
            ).get('elevation')

            topo_dict['TWI'] = twi_value.getInfo() if twi_value else np.nan

        except Exception as e:
            print(f"Ошибка при извлечении топографических данных: {e}")
            for key in ['elevation', 'slope', 'aspect', 'TWI']:
                topo_dict[key] = np.nan

        return topo_dict

    def extract_all(self, lon, lat):
        """Извлечение всех признаков для точки"""
        point = ee.Geometry.Point([lon, lat])

        features = {}
        features.update(self.get_sentinel_bands(point))
        features.update(self.get_indices(point))
        features.update(self.get_climate_data(point))
        features.update(self.get_topo_data(point))

        return features

    def process_points(self, df, lon_col='TH_LONG', lat_col='TH_LAT', output_path='lucas_features.csv'):
        """Обработка всех точек

        Параметры:
        -----------
        df : pandas.DataFrame
            DataFrame с данными точек
        lon_col : str
            Название колонки с долготой (по умолчанию 'TH_LONG')
        lat_col : str
            Название колонки с широтой (по умолчанию 'TH_LAT')
        output_path : str
            Путь для сохранения результата
        """
        print("\n" + "="*60)
        print("ИЗВЛЕЧЕНИЕ ПРИЗНАКОВ ИЗ GEE")
        print("="*60)

        # Проверяем наличие колонок с координатами
        if lon_col not in df.columns:
            raise ValueError(f"Колонка '{lon_col}' не найдена в DataFrame. Доступные колонки: {df.columns.tolist()}")
        if lat_col not in df.columns:
            raise ValueError(f"Колонка '{lat_col}' не найдена в DataFrame. Доступные колонки: {df.columns.tolist()}")

        print(f"\nВсего точек для обработки: {len(df)}")
        print(f"Используемые колонки координат: {lon_col}, {lat_col}")

        df_result = df.copy()

        # Инициализируем колонки для результатов
        feature_columns = ['B2', 'B3', 'B4', 'B8', 'B11', 'B12',
                          'BSF', 'CAL_mask', 'NDVI', 'ZPC1',
                          't_mean_annual', 't_max', 't_min', 'precip_annual',
                          'ETP', 'GTK',
                          'elevation', 'slope', 'aspect', 'TWI']

        for col in feature_columns:
            if col not in df_result.columns:
                df_result[col] = np.nan

        # Обработка точек
        for idx, row in tqdm(df.iterrows(), total=len(df), desc="Обработка точек"):
            try:
                lon = row[lon_col]
                lat = row[lat_col]

                if pd.isna(lon) or pd.isna(lat):
                    print(f"Точка {idx}: пропущены координаты")
                    continue

                features = self.extract_all(lon, lat)

                for key, value in features.items():
                    if key in df_result.columns:
                        df_result.loc[idx, key] = value

                time.sleep(0.1)

            except Exception as e:
                print(f"Ошибка для точки {idx} (координаты: {lon}, {lat}): {e}")

        df_result.to_csv(output_path, index=False)
        print(f"\nРезультат сохранен в {output_path}")
        print(f"Успешно обработано: {df_result[feature_columns].notna().any(axis=1).sum()} точек")

        return df_result




In [7]:
df_lucas = pd.read_csv(lucas_dataset)
print(f"Загружено {len(df_lucas)} точек")

df_sample = df_lucas.head(10)
print(f"Тестовая выборка: {len(df_sample)} точек")

extractor = FeatureExtractor()
df_final = extractor.process_points(df_sample, output_path='lucas_features.csv')

print("\nИтоговые колонки:")
print(df_final.columns.tolist())

print("\nПАЙПЛАЙН УСПЕШНО ЗАВЕРШЕН")

Загружено 18984 точек
Тестовая выборка: 10 точек

ИЗВЛЕЧЕНИЕ ПРИЗНАКОВ ИЗ GEE

Всего точек для обработки: 10
Используемые колонки координат: TH_LONG, TH_LAT


Обработка точек: 100%|██████████| 10/10 [04:22<00:00, 26.27s/it]


Результат сохранен в lucas_features.csv
Успешно обработано: 10 точек

Итоговые колонки:
['Depth', 'POINTID', 'pH_CaCl2', 'pH_H2O', 'EC', 'OC', 'CaCO3', 'P', 'N', 'K', 'OC (20-30 cm)', 'CaCO3 (20-30 cm)', 'Ox_Al', 'Ox_Fe', 'NUTS_0', 'NUTS_1', 'NUTS_2', 'NUTS_3', 'TH_LAT', 'TH_LONG', 'SURVEY_DATE', 'Elev', 'LC', 'LU', 'LC0_Desc', 'LC1_Desc', 'LU1_Desc', 'B2', 'B3', 'B4', 'B8', 'B11', 'B12', 'BSF', 'CAL_mask', 'NDVI', 'ZPC1', 't_mean_annual', 't_max', 't_min', 'precip_annual', 'ETP', 'GTK', 'elevation', 'slope', 'aspect', 'TWI']

ПАЙПЛАЙН УСПЕШНО ЗАВЕРШЕН
